# 🧾 템플릿 3 — 영수증 OCR + AI 가계부

영수증 사진 업로드하면 자동으로 글자 인식 → 가게/금액/카테고리 분류 → 누적 통계.

## 사용 기술
- **EasyOCR** (한국어/영어 무료 OCR)
- **LLM** (텍스트에서 가게/금액/카테고리 추출)
- **Pandas** (누적 통계)

⚠️ **GPU 권장**: 메뉴 → 런타임 → 런타임 유형 변경 → GPU (T4)
(CPU도 동작은 하지만 OCR이 느려집니다)

In [ ]:
!pip install -q easyocr gradio pandas openai

In [ ]:
# ===== 서버 연결 =====
SERVER_URL = "https://YOUR_URL.trycloudflare.com".strip().rstrip("/")
MODEL = "qwen2.5:7b-instruct"
assert "YOUR" not in SERVER_URL, "❌ SERVER_URL을 강사가 알려준 URL로 바꾸세요"

import httpx
from openai import OpenAI
client = OpenAI(base_url=f"{SERVER_URL}/v1", api_key="ollama",
                http_client=httpx.Client(headers={"User-Agent": "Mozilla/5.0"}))

# EasyOCR 초기화 (첫 실행 시 한국어 모델 ~50MB 다운로드)
import easyocr
print("EasyOCR 모델 로딩 중...")
reader = easyocr.Reader(['ko', 'en'], gpu=True)
print("✅ 준비 완료")

In [ ]:
import gradio as gr
import pandas as pd
import json
import re

# 누적 가계부 (브라우저 세션 내 유지)
ledger = {"items": []}

def extract_text(image):
    """영수증 이미지에서 텍스트 추출"""
    if image is None:
        return ""
    results = reader.readtext(image)
    return "\n".join(r[1] for r in results)

def parse_with_llm(text):
    """LLM이 영수증 텍스트에서 정보 뽑기"""
    prompt = f"""다음 영수증 텍스트에서 정보를 추출해 JSON으로만 답하세요. 다른 설명 없이 JSON만.

영수증 텍스트:
{text}

JSON 형식:
{{"store": "가게이름", "total": 금액숫자만, "category": "카테고리"}}

카테고리는 다음 중 하나: 음식점, 카페, 마트, 편의점, 쇼핑, 교통, 문화, 기타"""

    resp = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}],
        max_tokens=150, temperature=0.1,
    )
    out = resp.choices[0].message.content

    # JSON 추출 (LLM이 가끔 설명을 같이 줘서)
    m = re.search(r'\{[^\}]+\}', out, re.DOTALL)
    if m:
        try:
            return json.loads(m.group())
        except json.JSONDecodeError:
            pass
    return {"store": "(파싱 실패)", "total": 0, "category": "기타"}

def process_receipt(image):
    text = extract_text(image)
    if not text:
        return "영수증을 업로드하세요", pd.DataFrame(), "💰 누적 지출: 0원"

    parsed = parse_with_llm(text)
    parsed["raw_text"] = text[:200]
    ledger["items"].append(parsed)

    # 최근 입력 정보
    last = (
        f"### 📋 인식 결과\n"
        f"- **가게**: {parsed['store']}\n"
        f"- **금액**: {parsed['total']:,}원\n"
        f"- **카테고리**: {parsed['category']}\n\n"
        f"<details><summary>원본 텍스트 보기</summary>\n\n```\n{text}\n```\n</details>"
    )

    # 전체 내역 DataFrame
    df = pd.DataFrame([{k: v for k, v in i.items() if k != "raw_text"} for i in ledger["items"]])

    # 요약
    total = df["total"].sum()
    summary = f"## 💰 누적 지출: **{total:,}원** ({len(df)}회)\n\n### 카테고리별\n"
    by_cat = df.groupby("category")["total"].sum().sort_values(ascending=False)
    for cat, amt in by_cat.items():
        pct = amt / total * 100 if total > 0 else 0
        summary += f"- **{cat}**: {amt:,}원 ({pct:.0f}%)\n"

    return last, df, summary

def reset_ledger():
    ledger["items"] = []
    return None, "초기화됨", pd.DataFrame(), "💰 누적 지출: 0원"

with gr.Blocks(title="🧾 AI 가계부", theme=gr.themes.Soft()) as demo:
    gr.Markdown("# 🧾 영수증 OCR + AI 가계부")
    gr.Markdown("영수증 사진을 업로드하면 자동으로 가계부에 정리됩니다.")
    
    with gr.Row():
        with gr.Column():
            img = gr.Image(type="numpy", label="영수증 사진", height=400)
            btn = gr.Button("➤ 영수증 분석", variant="primary", size="lg")
            reset_btn = gr.Button("🔄 가계부 초기화")
            last = gr.Markdown()
        with gr.Column():
            summary = gr.Markdown("💰 누적 지출: 0원")
            history = gr.Dataframe(label="전체 내역", interactive=False)

    btn.click(process_receipt, img, [last, history, summary])
    reset_btn.click(reset_ledger, outputs=[img, last, history, summary])

demo.launch(share=True)

---
## 🚀 바이브 코딩 확장 아이디어

### 쉬움
- 월별 / 일별 합산
- 카테고리 자동 분류 정교화 (LLM 프롬프트 개선)
- 예산 설정 ("이번 달 50만원 한도") + 초과 알림

### 중간
- 카테고리별 막대그래프 / 파이차트
- 시간대별 소비 패턴
- 영수증 사진 ZIP 일괄 업로드
- 비슷한 가게 자동 그룹화 ("스타벅스" 변형들 다 카페로)

### 도전적
- 데이터 영구 저장 (Google Sheets API 연동)
- 친구들과 비용 정산 (더치페이 자동 계산)
- 영수증에서 개별 품목까지 추출 (커피, 디저트 등)
- 이전 카드사 명세서 OCR로 가져오기

### 🎁 자랑하기 팁
- 한 달 영수증 30장 정리해서 "이번 달 내 소비" 인포그래픽으로 만들기
- 친구 단톡방용 "오늘 더치페이" 도구로 발전